## Agregacija

In [2]:
import pandas as pd

student_info = pd.read_csv("data/raw/studentInfo.csv")
student_assessment = pd.read_csv("data/raw/studentAssessment.csv")
assessments = pd.read_csv("data/raw/assessments.csv")
student_vle = pd.read_csv("data/raw/studentVle.csv")
student_registration = pd.read_csv("data/raw/studentRegistration.csv")
vle=pd.read_csv("data/raw/vle.csv")
courses=pd.read_csv("data/raw/courses.csv")

In [3]:
#1.Prosecna ocena 
avg_score= student_assessment.groupby("id_student")["score"].mean().reset_index()
avg_score.columns=["id_student", "avg_score"]
avg_score.head()

,id_student,avg_score
0,6516,61.800000
1,8462,87.000000
2,11391,82.000000
3,23629,82.500000
4,23698,74.444444


In [4]:
#2.Standardna devijacija ocena 
std_score=student_assessment.groupby("id_student")["score"].std().reset_index()
std_score.columns=["id_student", "std_score"]
std_score.head(5)

,id_student,std_score
0,6516,10.329569
1,8462,4.472136
2,11391,3.082207
3,23629,20.273135
4,23698,13.258121


In [5]:
#3.Odnos predatih i ukupnih zadataka
count_submitted=student_assessment.groupby("id_student")["id_assessment"].count().reset_index()
count_submitted.columns=["id_student", "count_submitted"]
count_submitted.sample(5)

,id_student,count_submitted
22581,2487430,5
549,109155,5
22484,2457907,5
5884,505754,5
16219,630109,12


In [6]:
total_assessments=assessments.groupby(["code_module","code_presentation"])["id_assessment"].count().reset_index()
total_assessments.columns=["code_module", "code_presentation", "total_assessments"]
total_assessments

,code_module,code_presentation,total_assessments
0,AAA,2013J,6
1,AAA,2014J,6
2,BBB,2013B,12
3,BBB,2013J,12
4,BBB,2014B,12
5,BBB,2014J,6
6,CCC,2014B,10
7,CCC,2014J,10
8,DDD,2013B,14
9,DDD,2013J,7


In [7]:
submission_rate=count_submitted.merge(
    student_info[["id_student","code_module", "code_presentation"]],
    on="id_student",
    how="left"
)
submission_rate.head()

,id_student,count_submitted,code_module,code_presentation
0,6516,5,AAA,2014J
1,8462,7,DDD,2013J
2,8462,7,DDD,2014J
3,11391,5,AAA,2013J
4,23629,4,BBB,2013B


In [8]:
submission_rate=submission_rate.merge(
    total_assessments,
    on=["code_module", "code_presentation"],
    how="left"
)

submission_rate.head(5)

,id_student,count_submitted,code_module,code_presentation,total_assessments
0,6516,5,AAA,2014J,6
1,8462,7,DDD,2013J,7
2,8462,7,DDD,2014J,7
3,11391,5,AAA,2013J,6
4,23629,4,BBB,2013B,12


In [9]:
submission_rate["submission_rate"]=submission_rate["count_submitted"]/submission_rate["total_assessments"]
submission_rate.sample(10)

,id_student,count_submitted,code_module,code_presentation,total_assessments,submission_rate
14339,596326,2,EEE,2014J,5,0.400000
4008,395715,5,BBB,2013J,12,0.416667
11840,575811,5,DDD,2013J,7,0.714286
19645,634536,21,FFF,2014B,13,1.615385
21421,655890,9,CCC,2014J,10,0.900000
951,141648,12,FFF,2013J,13,0.923077
13733,592007,21,FFF,2013J,13,1.615385
23744,698362,5,BBB,2014J,6,0.833333
21134,652801,3,FFF,2014J,13,0.230769
411,80717,11,BBB,2013J,12,0.916667


In [10]:
assessments["assessment_type"].value_counts()

assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64

In [11]:
total_assessments=assessments[assessments["assessment_type"]!="Exam"].groupby( ["code_module", "code_presentation"])["id_assessment"].count().reset_index()
total_assessments.columns = ["code_module", "code_presentation", "total_assessments"]
total_assessments

,code_module,code_presentation,total_assessments
0,AAA,2013J,5
1,AAA,2014J,5
2,BBB,2013B,11
3,BBB,2013J,11
4,BBB,2014B,11
5,BBB,2014J,5
6,CCC,2014B,8
7,CCC,2014J,8
8,DDD,2013B,13
9,DDD,2013J,6


In [12]:
submission_rate = count_submitted.merge(
    student_info[["id_student", "code_module", "code_presentation"]],
    on="id_student",
    how="left"
)

submission_rate = submission_rate.merge(
    total_assessments,
    on=["code_module", "code_presentation"],
    how="left"
)

submission_rate["submission_rate"] = submission_rate["count_submitted"] / submission_rate["total_assessments"]
submission_rate = submission_rate[["id_student", "submission_rate"]]
submission_rate.sample(10)

,id_student,submission_rate
300,66451,1.000000
19544,634139,1.500000
6213,494896,0.909091
25869,2494700,1.000000
2601,311474,0.333333
5921,486099,0.909091
21012,651507,1.166667
21373,655242,1.000000
12358,581200,1.000000
17618,623081,0.888889


In [13]:
problem = submission_rate[submission_rate["submission_rate"] > 1]
print("Broj studenata sa rate > 1:", len(problem))
problem.head()

Broj studenata sa rate > 1: 6155


,id_student,submission_rate
1,8462,1.166667
2,8462,1.166667
5,23698,1.125000
9,24213,1.166667
23,27116,1.125000


In [14]:
student_info[student_info["id_student"]==8462][["id_student","code_module","code_presentation"]]

,id_student,code_module,code_presentation
14395,8462,DDD,2013J
17560,8462,DDD,2014J


In [15]:
student_assessment_full=student_assessment.merge(
    assessments[["id_assessment","code_module","code_presentation", "date"]],
    on="id_assessment",
    how="left"
)
student_assessment_full.head()

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,date
0,1752,11391,18,0,78.0,AAA,2013J,19.0
1,1752,28400,22,0,70.0,AAA,2013J,19.0
2,1752,31604,17,0,72.0,AAA,2013J,19.0
3,1752,32885,26,0,69.0,AAA,2013J,19.0
4,1752,38053,19,0,79.0,AAA,2013J,19.0


In [16]:
avg_score = student_assessment_full.groupby(["id_student", "code_module", "code_presentation"])["score"].mean().reset_index()
avg_score.columns = ["id_student", "code_module", "code_presentation", "avg_score"]
avg_score.head()

,id_student,code_module,code_presentation,avg_score
0,6516,AAA,2014J,61.800000
1,8462,DDD,2013J,87.666667
2,8462,DDD,2014J,86.500000
3,11391,AAA,2013J,82.000000
4,23629,BBB,2013B,82.500000


In [17]:
std_score = student_assessment_full.groupby(["id_student", "code_module", "code_presentation"])["score"].std().reset_index()
std_score.columns = ["id_student", "code_module", "code_presentation", "std_score"]
std_score.head()

,id_student,code_module,code_presentation,std_score
0,6516,AAA,2014J,10.329569
1,8462,DDD,2013J,5.033223
2,8462,DDD,2014J,4.725816
3,11391,AAA,2013J,3.082207
4,23629,BBB,2013B,20.273135


In [18]:
non_exam = assessments[assessments["assessment_type"] != "Exam"]["id_assessment"]
student_assessment_no_exam = student_assessment_full[student_assessment_full["id_assessment"].isin(non_exam)]
student_assessment_no_exam.head()

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,date
0,1752,11391,18,0,78.0,AAA,2013J,19.0
1,1752,28400,22,0,70.0,AAA,2013J,19.0
2,1752,31604,17,0,72.0,AAA,2013J,19.0
3,1752,32885,26,0,69.0,AAA,2013J,19.0
4,1752,38053,19,0,79.0,AAA,2013J,19.0


In [19]:
count_submitted = student_assessment_no_exam.groupby(["id_student", "code_module", "code_presentation"])["id_assessment"].count().reset_index()
count_submitted.columns = ["id_student", "code_module", "code_presentation", "count_submitted"]
count_submitted.head()

,id_student,code_module,code_presentation,count_submitted
0,6516,AAA,2014J,5
1,8462,DDD,2013J,3
2,8462,DDD,2014J,4
3,11391,AAA,2013J,5
4,23629,BBB,2013B,4


In [20]:
submission_rate = count_submitted.merge(
    total_assessments,
    on=["code_module", "code_presentation"],
    how="left"
)

submission_rate["submission_rate"] = submission_rate["count_submitted"] / submission_rate["total_assessments"]
submission_rate = submission_rate[["id_student", "code_module", "code_presentation", "submission_rate"]]

print("Vrednosti preko 1:", len(submission_rate[submission_rate["submission_rate"] > 1]))
submission_rate.head()

Vrednosti preko 1: 0


,id_student,code_module,code_presentation,submission_rate
0,6516,AAA,2014J,1.000000
1,8462,DDD,2013J,0.500000
2,8462,DDD,2014J,0.666667
3,11391,AAA,2013J,1.000000
4,23629,BBB,2013B,0.363636


In [21]:
#4. Prosecan broj dana od pre roka predaje 
student_assessment_full["days_before_deadline"]=student_assessment_full["date"]-student_assessment_full["date_submitted"]

avg_days_before_deadline=student_assessment_full.groupby(["id_student", "code_module", "code_presentation"])["days_before_deadline"].mean().reset_index()
avg_days_before_deadline.columns=["id_student", "code_module", "code_presentation", "avg_days_before_deadline"]
avg_days_before_deadline["avg_days_before_deadline"] = (avg_days_before_deadline["avg_days_before_deadline"].round(2))
avg_days_before_deadline.head()

,id_student,code_module,code_presentation,avg_days_before_deadline
0,6516,AAA,2014J,2.60
1,8462,DDD,2013J,0.33
2,8462,DDD,2014J,59.50
3,11391,AAA,2013J,1.80
4,23629,BBB,2013B,-3.50


In [22]:
#5. Ukupan broj klikova po studentu
total_clicks=student_vle.groupby(["id_student", "code_module", "code_presentation"])["sum_click"].sum().reset_index()
total_clicks.columns = ["id_student", "code_module", "code_presentation", "total_clicks"]
total_clicks.head()

,id_student,code_module,code_presentation,total_clicks
0,6516,AAA,2014J,2791
1,8462,DDD,2013J,646
2,8462,DDD,2014J,10
3,11391,AAA,2013J,934
4,23629,BBB,2013B,161


In [23]:
#6.Poslednji dan kada je student bio aktivan
last_active_day = student_vle.groupby(["id_student", "code_module", "code_presentation"])["date"].max().reset_index()
last_active_day.columns = ["id_student", "code_module", "code_presentation", "last_active_day"]
last_active_day.head()

,id_student,code_module,code_presentation,last_active_day
0,6516,AAA,2014J,269
1,8462,DDD,2013J,118
2,8462,DDD,2014J,10
3,11391,AAA,2013J,253
4,23629,BBB,2013B,87


In [24]:
student_vle_full = student_vle.merge(
    vle[["id_site", "activity_type"]],
    on="id_site",
    how="left"
)
student_vle_full.head()

,code_module,code_presentation,id_student,id_site,date,sum_click,activity_type
0,AAA,2013J,28400,546652,-10,4,forumng
1,AAA,2013J,28400,546652,-10,1,forumng
2,AAA,2013J,28400,546652,-10,1,forumng
3,AAA,2013J,28400,546614,-10,11,homepage
4,AAA,2013J,28400,546714,-10,1,oucontent


In [25]:
#7. Broj razlicitih tipova aktivnosti
unique_activities = student_vle_full.groupby(["id_student", "code_module", "code_presentation"])["activity_type"].nunique().reset_index()
unique_activities.columns = ["id_student", "code_module", "code_presentation", "unique_activities"]
unique_activities.head()

,id_student,code_module,code_presentation,unique_activities
0,6516,AAA,2014J,7
1,8462,DDD,2013J,9
2,8462,DDD,2014J,3
3,11391,AAA,2013J,6
4,23629,BBB,2013B,5


In [26]:
df = student_info.copy()

for agregacija in [avg_score, std_score, submission_rate, 
                   avg_days_before_deadline, total_clicks, 
                   last_active_day, unique_activities]:
    df = df.merge(agregacija, on=["id_student", "code_module", "code_presentation"], how="left")

print("Shape:", df.shape)
df.head()

Shape: (32593, 19)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,avg_score,std_score,submission_rate,avg_days_before_deadline,total_clicks,last_active_day,unique_activities
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,82.0,3.082207,1.0,1.8,934.0,253.0,6.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,66.4,4.335897,1.0,0.0,1435.0,239.0,7.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,NaN,NaN,NaN,NaN,281.0,12.0,6.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,76.0,6.892024,1.0,2.0,2158.0,264.0,8.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,54.4,20.513410,1.0,-11.4,1034.0,247.0,7.0


In [27]:
df[["avg_score", "std_score", "submission_rate", 
    "avg_days_before_deadline", "total_clicks", 
    "last_active_day", "unique_activities"]].isnull().sum()

avg_score                   6773
std_score                   9292
submission_rate             6754
avg_days_before_deadline    6751
total_clicks                3365
last_active_day             3365
unique_activities           3365
dtype: int64

## Odredjivanje granice

In [28]:
courses["module_presentation_length"].describe()

count     22.000000
mean     255.545455
std       13.654677
min      234.000000
25%      241.000000
50%      261.500000
75%      268.000000
max      269.000000
Name: module_presentation_length, dtype: float64

In [29]:
student_assessment["date_submitted"].describe()

count    173912.000000
mean        116.032942
std          71.484148
min         -11.000000
25%          51.000000
50%         116.000000
75%         173.000000
max         608.000000
Name: date_submitted, dtype: float64

In [30]:
student_vle["date"].describe()

count    1.065528e+07
mean     9.517400e+01
std      7.607130e+01
min     -2.500000e+01
25%      2.500000e+01
50%      8.600000e+01
75%      1.560000e+02
max      2.690000e+02
Name: date, dtype: float64

In [31]:
GRANICA=90
student_assessment_filtered=student_assessment_full[student_assessment_full["date_submitted"]<=GRANICA]
student_vle_filtered=student_vle_full[student_vle_full["date"]<=GRANICA]

print("studentAssessment pre filtriranja:", len(student_assessment_full))
print("studentAssessment posle filtriranja:", len(student_assessment_filtered))
print("studentVle pre filtriranja:", len(student_vle_full))
print("studentVle posle filtriranja:", len(student_vle_filtered))

studentAssessment pre filtriranja: 173912
studentAssessment posle filtriranja: 71051
studentVle pre filtriranja: 10655280
studentVle posle filtriranja: 5486878


In [32]:
avg_score_90 = student_assessment_filtered.groupby(["id_student", "code_module", "code_presentation"])["score"].mean().reset_index()
avg_score_90.columns = ["id_student", "code_module", "code_presentation", "avg_score"]
avg_score_90.head()

,id_student,code_module,code_presentation,avg_score
0,6516,AAA,2014J,54.000000
1,8462,DDD,2013J,87.666667
2,8462,DDD,2014J,86.500000
3,11391,AAA,2013J,81.500000
4,23629,BBB,2013B,76.666667


In [33]:
std_score_90 = student_assessment_filtered.groupby(["id_student", "code_module", "code_presentation"])["score"].std().reset_index()
std_score_90.columns = ["id_student", "code_module", "code_presentation", "std_score"]
std_score_90.head()

,id_student,code_module,code_presentation,std_score
0,6516,AAA,2014J,8.485281
1,8462,DDD,2013J,5.033223
2,8462,DDD,2014J,4.725816
3,11391,AAA,2013J,4.949747
4,23629,BBB,2013B,20.305993


In [34]:
assessments_no_exam_90 = assessments[(assessments["id_assessment"].isin(non_exam)) &(assessments["date"] <= GRANICA)]
total_assessments_90 = assessments_no_exam_90.groupby(["code_module", "code_presentation"])["id_assessment"].count().reset_index()
total_assessments_90.columns = ["code_module","code_presentation","total_assessments_90"]
total_assessments_90.head()


,code_module,code_presentation,total_assessments_90
0,AAA,2013J,2
1,AAA,2014J,2
2,BBB,2013B,5
3,BBB,2013J,3
4,BBB,2014B,5


In [35]:
eligible_assessments_90 = assessments_no_exam_90["id_assessment"]

student_assessment_no_exam_90 = student_assessment_filtered[student_assessment_filtered["id_assessment"].isin(eligible_assessments_90)]
count_submitted_90 = student_assessment_no_exam_90.groupby(["id_student", "code_module", "code_presentation"])["id_assessment"].count().reset_index(name="count_submitted")
count_submitted_90.head()

,id_student,code_module,code_presentation,count_submitted
0,6516,AAA,2014J,2
1,8462,DDD,2013J,3
2,8462,DDD,2014J,3
3,11391,AAA,2013J,2
4,23629,BBB,2013B,3


In [36]:
submission_rate_90 = count_submitted_90.merge(
    total_assessments_90,
    on=["code_module", "code_presentation"],
    how="left"
)

submission_rate_90["submission_rate_90"] = submission_rate_90["count_submitted"] /submission_rate_90["total_assessments_90"]
submission_rate_90.head()

,id_student,code_module,code_presentation,count_submitted,total_assessments_90,submission_rate_90
0,6516,AAA,2014J,2,2,1.0
1,8462,DDD,2013J,3,3,1.0
2,8462,DDD,2014J,3,3,1.0
3,11391,AAA,2013J,2,2,1.0
4,23629,BBB,2013B,3,5,0.6


In [37]:
total_clicks_90 = student_vle_filtered.groupby(["id_student", "code_module", "code_presentation"])["sum_click"].sum().reset_index()
total_clicks_90.columns = ["id_student", "code_module", "code_presentation", "total_clicks"]
total_clicks_90.head()

,id_student,code_module,code_presentation,total_clicks
0,6516,AAA,2014J,1296
1,8462,DDD,2013J,607
2,8462,DDD,2014J,10
3,11391,AAA,2013J,587
4,23629,BBB,2013B,161


In [38]:
last_active_90 = student_vle_filtered.groupby(["id_student", "code_module", "code_presentation"])["date"].max().reset_index()
last_active_90.columns = ["id_student", "code_module", "code_presentation", "last_active_day"]
last_active_90.head()

,id_student,code_module,code_presentation,last_active_day
0,6516,AAA,2014J,87
1,8462,DDD,2013J,90
2,8462,DDD,2014J,10
3,11391,AAA,2013J,87
4,23629,BBB,2013B,87


In [39]:
avg_days_90 = student_assessment_filtered.groupby(["id_student", "code_module", "code_presentation"])["days_before_deadline"].mean().reset_index()
avg_days_90.columns = ["id_student", "code_module", "code_presentation", "avg_days_before_deadline"]
avg_days_90.head()

,id_student,code_module,code_presentation,avg_days_before_deadline
0,6516,AAA,2014J,2.500000
1,8462,DDD,2013J,0.333333
2,8462,DDD,2014J,59.500000
3,11391,AAA,2013J,1.000000
4,23629,BBB,2013B,-2.666667


In [40]:
unique_activities_90 = student_vle_filtered.groupby(["id_student", "code_module", "code_presentation"])["activity_type"].nunique().reset_index()
unique_activities_90.columns = ["id_student", "code_module", "code_presentation", "unique_activities"]
unique_activities_90.head()

,id_student,code_module,code_presentation,unique_activities
0,6516,AAA,2014J,7
1,8462,DDD,2013J,9
2,8462,DDD,2014J,3
3,11391,AAA,2013J,6
4,23629,BBB,2013B,5


In [41]:
df_90 = student_info.copy()

for agregacija in [avg_score_90, std_score_90, submission_rate_90, 
                   avg_days_90, total_clicks_90, 
                   last_active_90, unique_activities_90]:
    df_90 = df_90.merge(agregacija, on=["id_student", "code_module", "code_presentation"], how="left")

print("Shape:", df_90.shape)
df_90.head()

Shape: (32593, 21)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,final_result,avg_score,std_score,count_submitted,total_assessments_90,submission_rate_90,avg_days_before_deadline,total_clicks,last_active_day,unique_activities
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,Pass,81.5,4.949747,2.0,2.0,1.0,1.0,587.0,87.0,6.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,Pass,69.0,1.414214,2.0,2.0,1.0,-0.5,821.0,90.0,6.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,Withdrawn,NaN,NaN,NaN,NaN,NaN,NaN,281.0,12.0,6.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,Pass,71.5,0.707107,2.0,2.0,1.0,2.5,992.0,90.0,7.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,Pass,49.5,27.577164,2.0,2.0,1.0,-14.0,684.0,90.0,7.0


In [42]:
student_registration.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


## Izbacivanje studenata koji su se ispisali pre granice

In [43]:
student_registration_filter = student_registration[(student_registration["date_unregistration"].isna()) | (student_registration["date_unregistration"] > 90)]
print("Pre filtriranja:", len(student_registration))
print("Posle filtriranja:", len(student_registration_filter))

Pre filtriranja: 32593
Posle filtriranja: 25562


In [44]:
df_90_final = df_90.merge(
    student_registration_filter[["id_student", "code_module", "code_presentation"]],
    on=["id_student", "code_module", "code_presentation"],
    how="inner"
)

print("df_90 pre:", len(df_90))
print("df_90 posle:", len(df_90_final))

df_90 pre: 32593
df_90 posle: 25562


In [45]:
df_90_final.to_csv("data/raw/student_info_final.csv", index=False)

## Funkcija i pivot tabele 

In [ ]:
def napravi_agregacije(granica):
    print("Granica:", granica)

    student_registration_filter=student_registration[(student_registration["date_unregistration"].isna()) | (student_registration["date_unregistration"]>granica)]
    
    df_granica=student_info.merge(
        student_registration_filter[["id_student", "code_module", "code_presentation"]],
        on=["id_student", "code_module", "code_presentation"],
        how="inner"
    )
    
    student_assessment_full=student_assessment.merge(
        assessments,
        on="id_assessment",
        how="left"
    )
    
    student_assessment_filtered=student_assessment_full[student_assessment_full["date_submitted"]<=granica].copy()
    
    assessments_no_exam_until=assessments[(assessments["assessment_type"]!="Exam") & (assessments["date"]<=granica)].copy()
    
    eligible_assessments=assessments_no_exam_until["id_assessment"]
    
    student_assessment_no_exam=student_assessment_filtered[student_assessment_filtered["id_assessment"].isin(eligible_assessments)].copy()
    
    student_assessment_no_exam["days_before_deadline"]=student_assessment_no_exam["date"]-student_assessment_no_exam["date_submitted"]
     
    assessment_agg=(
        student_assessment_no_exam
        .groupby([
            "id_student",
            "code_module",
            "code_presentation",
            "assessment_type"
        ])
        .agg(
            avg_score=("score", "mean"),
            std_score=("score", "std"),
            count_submitted=("id_assessment", "nunique"),
            avg_days_before_deadline=("days_before_deadline", "mean")
        )
        .reset_index()
    )
    
    assessment_pivot=assessment_agg.pivot_table(
        index=["id_student", "code_module", "code_presentation"],
        columns="assessment_type",
        values=[
            "avg_score",
            "std_score",
            "count_submitted",
            "avg_days_before_deadline"
        ]
    )
    
    assessment_pivot.columns=[f"{kolona[0]}_{kolona[1]}"for kolona in assessment_pivot.columns]
    
    assessment_pivot=assessment_pivot.reset_index()
    
    total_assessments_by_type=assessments_no_exam_until.groupby(["code_module", "code_presentation", "assessment_type"])["id_assessment"].nunique().reset_index(name="total_assessments")

    total_assessments_pivot=total_assessments_by_type.pivot_table(
        index=["code_module", "code_presentation"],
        columns="assessment_type",
        values="total_assessments",
        fill_value=0
    )
    
    total_assessments_pivot.columns=[f"total_assessments_{kolona}" for kolona in total_assessments_pivot.columns]
    
    total_assessments_pivot=total_assessments_pivot.reset_index()
    
    assessment_pivot=assessment_pivot.merge(
        total_assessments_pivot,
        on=["code_module", "code_presentation"],
        how="left"
    )
    
    for tip in ["TMA", "CMA"]:
        count_col=f"count_submitted_{tip}"
        total_col=f"total_assessments_{tip}"
        rate_col=f"submission_rate_{tip}"
        
        if count_col in assessment_pivot.columns and total_col in assessment_pivot.columns:
            assessment_pivot[rate_col]=(assessment_pivot[count_col] / assessment_pivot[total_col])
    
    student_vle_full=student_vle.merge(
        vle,
        on=["id_site", "code_module", "code_presentation"],
        how="left"
    )
    
    student_vle_filtered=student_vle_full[student_vle_full["date"]<=granica].copy()
    
    total_clicks=student_vle_filtered.groupby(["id_student", "code_module", "code_presentation"])["sum_click"].sum().reset_index()
    
    total_clicks.columns=[
        "id_student",
        "code_module",
        "code_presentation",
        "total_clicks"
    ]
    
    last_active=student_vle_filtered.groupby(["id_student", "code_module", "code_presentation"])["date"].max().reset_index()
    
    last_active.columns=[
        "id_student",
        "code_module",
        "code_presentation",
        "last_active_day"
    ]
    
    unique_activities=student_vle_filtered.groupby(["id_student", "code_module", "code_presentation"])["activity_type"].nunique().reset_index()
    
    unique_activities.columns=[
        "id_student",
        "code_module",
        "code_presentation",
        "unique_activities"
    ]
    
    for agregacija in [assessment_pivot, total_clicks, last_active, unique_activities]:
        df_granica=df_granica.merge(
            agregacija,
            on=["id_student", "code_module", "code_presentation"],
            how="left"
        )
   
    numericke_kolone = df_granica.select_dtypes(include="number").columns

    df_granica[numericke_kolone] = (df_granica[numericke_kolone].fillna(0))

    df_granica["imd_band"] = (df_granica["imd_band"].fillna("Unknown"))
    
    print(df_granica.shape)

    df_granica.to_csv("data/processed/final.csv",index=False)
    print(df_granica.shape)
    print(f"Podaci za granicu od {granica} dana sačuvani su u data/processed/final.csv")
    
    return df_granica




In [54]:
df=napravi_agregacije(90)
df.head()

Granica: 90
(25562, 27)
(25562, 27)
Podaci za granicu od 90 dana sačuvani su u data/raw/final.csv


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,count_submitted_TMA,std_score_CMA,std_score_TMA,total_assessments_CMA,total_assessments_TMA,submission_rate_TMA,submission_rate_CMA,total_clicks,last_active_day,unique_activities
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,2.0,0.0,4.949747,0.0,2.0,1.0,0.0,587.0,87.0,6.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,2.0,0.0,1.414214,0.0,2.0,1.0,0.0,821.0,90.0,6.0
2,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,2.0,0.0,0.707107,0.0,2.0,1.0,0.0,992.0,90.0,7.0
3,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,2.0,0.0,27.577164,0.0,2.0,1.0,0.0,684.0,90.0,7.0
4,AAA,2013J,38053,M,Wales,A Level or Equivalent,80-90%,35-55,0,60,...,2.0,0.0,7.071068,0.0,2.0,1.0,0.0,1060.0,86.0,7.0
